# ComfyUI Master Pipeline on Google Colab Pro
### SDXL & FLUX.1 ハイブリッド完全自動化パイプライン

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/houjiro/googlecolab-by-ai/blob/main/ComfyUI_Colab_Master.ipynb)

Google Colab ProのリソースとGoogle Driveを連携させ、**SDXL** および **FLUX.1 [dev]** をワンタッチで切り替えて自律生成を行うための統合パイプラインです。

## 0. リポジトリの同期
最新スクリプト（FLUX対応版）を作業ディレクトリに同期します。

In [ ]:
import os
if not os.path.exists("/content/googlecolab-by-ai"):
    !git clone https://github.com/houjiro/googlecolab-by-ai.git /content/googlecolab-by-ai
else:
    !git -C /content/googlecolab-by-ai pull
%cd /content/googlecolab-by-ai

## 1. 【フェーズ1】環境初期化・高速起動
- Google Driveマウント
- Miniconda (Python 3.10) 仮想環境の展開/構築（キャッシュ復元対応）
- ComfyUI / PyTorch の整合性確認

In [ ]:
%run setup_comfyui.py

## 2. 【フェーズ1・2】ComfyUI サーバーの起動 (バックグラウンド)
- ローカルAPI (http://127.0.0.1:8188) として起動
- 生成画像は自動的に Google Drive の `outputs` フォルダに保存されます

In [ ]:
import subprocess
import time
import urllib.request
from setup_comfyui import DRIVE_MASTER_DIR

conda_python = "/content/miniconda/envs/comfy_env/bin/python"
comfy_dir = "/content/ComfyUI"
output_dir = str(DRIVE_MASTER_DIR / "outputs")

print(f"Starting ComfyUI server in the background (Output: {output_dir})...")
comfy_cmd = [
    conda_python,
    f"{comfy_dir}/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--output-directory", output_dir,
    "--preview-method", "auto",
    "--highvram"
]

with open("/content/comfyui.log", "w") as log_file:
    comfy_proc = subprocess.Popen(comfy_cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("Waiting for ComfyUI to become ready...")
server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

if server_ready:
    print("\n[SUCCESS] ComfyUI server is LIVE at http://127.0.0.1:8188 !")
else:
    print("\n[WARN] Server check timed out. Last 30 lines of log:")
    !tail -n 30 /content/comfyui.log

## 3. 【フェーズ2・3】APIラッパー関数のロード
SDXLおよびFLUXの両方に対応した自律生成関数を読み込みます。

In [ ]:
from pipeline_tools import (
    install_model,
    install_custom_node,
    generate_media,
    preview_image,
    create_sdxl_txt2img_workflow,
    create_sdxl_lora_workflow,
    create_flux_workflow,
    DRIVE_MASTER_DIR
)
print(f"[OK] Pipeline tools loaded (SDXL & FLUX.1 Ready). Drive Master: {DRIVE_MASTER_DIR}")

## 4-A. 【SDXL】モデルのダウンロード (初回のみ / 既にDL済みならスキップ可)

In [ ]:
# SDXL Base 1.0 & VAE のダウンロード
%run download_sdxl.py

## 4-B. 【FLUX.1】スイートのダウンロード (初回のみ)
FLUX.1 [dev] FP8 (~11.9GB)、CLIP-L、T5-XXL FP8、専用VAE を Google Drive に高速ダウンロードします。

In [ ]:
# FLUX.1 [dev] スイートのダウンロード
%run download_flux.py

## 5. 【生成実行】FLUX.1 + LoRA (nadeshiko_jats) 連続生成
Seedream 5 Pro級の最高峰画質とプロンプト理解力で画像を生成します。

In [ ]:
import time
TOTAL_COUNT = 1 # まずは1枚でテスト（何枚でも指定可能）

positive_prompt = """Photorealistic 35mm color photograph, authentic travel documentary photo, 16:9 aspect ratio, completely realistic live-action photo, not an illustration, not anime, not 3D render. In the breathtaking volcanic landscape of Kusasenrigahama in Mount Aso, a gorgeous Japanese woman named Nadeshiko in her 20s with long flowing black hair and radiant smile, wearing stylish earth-toned outdoor travel jacket and cargo pants, standing on meadow pathway with dramatic mountain backdrop."""

for i in range(1, TOTAL_COUNT + 1):
    current_seed = int(time.time() * 1000 + i * 9973) % 10000000000
    print(f"\n▶ FLUX.1 生成中 [{i}/{TOTAL_COUNT}] (Seed: {current_seed})...")
    workflow = create_flux_workflow(
        positive_prompt=positive_prompt,
        lora_name="nadeshiko_jats.safetensors",
        lora_strength=0.85,
        width=1344,
        height=768,
        steps=25,
        guidance=3.5,
        seed=current_seed
    )
    output_images = generate_media(workflow)
    for img in output_images:
        preview_image(img)